# PyPSA-Earth Netzwerkanalyse: Vergleich zweier Szenarien

Diese Vorlage lädt zwei PyPSA-Earth-Netzwerke ein und bereitet eine saubere Basis für spätere Auswertungen vor.

Die Notebook-Struktur ist bewusst schlank gehalten:
1. Pakete importieren  
2. zentrale Pfade definieren  
3. zwei Netzwerke laden  
4. optionale Zusatzdateien laden  
5. kurze Plausibilitätsübersicht anzeigen  

Weitere Analyse- und Plot-Blöcke können später darunter eingefügt werden.

## 1. Pakete importieren

In [1]:
import yaml
import pypsa
import warnings
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
from datetime import datetime
from cartopy import crs as ccrs
from pypsa.plot import add_legend_circles, add_legend_lines, add_legend_patches
import os
import xarray as xr
import cartopy
import glob
import logging
import cartopy.crs as ccrs
import fiona

# ------------------------------------------------------------
# Notebook-Einstellungen
# ------------------------------------------------------------
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

logging.getLogger("pypsa.io").setLevel(logging.ERROR)

plt.rcParams["figure.dpi"] = 120
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

## 2. Zentraler Pfadblock

Hier werden alle Pfade eingetragen, die für die Analyse benötigt werden.

Wichtig:
- `NETWORK_PATH_SCENARIO_A` und `NETWORK_PATH_SCENARIO_B` sind Pflicht.
- Alle anderen Dateien sind optional und können leer bleiben, falls sie für die spätere Analyse nicht benötigt werden.
- Die Pfade können als Windows-WSL-Pfade (`/mnt/c/...`) oder Linux-Pfade eingetragen werden.

In [2]:
# ============================================================
# ZENTRALER PFADBLOCK
# ============================================================

# Projektordner deines PyPSA-Earth-Forks
PROJECT_ROOT = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal")

# Namen der Szenarien für Tabellen, Plots und spätere Vergleiche
SCENARIO_A_NAME = "EGS"
SCENARIO_B_NAME = "without EGS"

# ------------------------------------------------------------
# Pflicht: Netzwerkdateien
# ------------------------------------------------------------
#EGS
NETWORK_PATH_SCENARIO_A = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/EGS_Test/postnetworks/elec_s_10_ec_lcopt_Co2L0.0-Ep_144h_2050_0.0514_AG_0export_393import.nc")
#without EGS
NETWORK_PATH_SCENARIO_B = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/Without_EGS_Test/postnetworks/elec_s_10_ec_lcopt_Co2L0.0-Ep_144h_2050_0.0514_AG_0export_393import.nc")

# ------------------------------------------------------------
# Optional: Config-Dateien
# ------------------------------------------------------------

CONFIG_PATH_SCENARIO_A = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/SCENARIO_A/configs/config.yaml")

CONFIG_PATH_SCENARIO_B = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/SCENARIO_B/configs/config.yaml")

# ------------------------------------------------------------
# Optional: Geodaten
# ------------------------------------------------------------

COUNTRY_SHAPES_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/EGS_Test/shapes/country_shapes.geojson")

GADM_SHAPES_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/data/gadm/gadm41_JPN/gadm41_JPN.gpkg")

GADM_SHAPES_PATH_10 = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/data/gadm/gadm41_JPN/old/OUTPUT.gpkg")

REGIONS_ONSHORE_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/EGS_Test/bus_regions/regions_onshore_elec_s_10.geojson")

REGIONS_OFFSHORE_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/EGS_Test/bus_regions/regions_offshore_elec_s_10.geojson")

# ------------------------------------------------------------
# Optional: Profil- und Ressourcen-Dateien
# ------------------------------------------------------------

SOLAR_PROFILE_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/EGS_Test/renewable_profiles/profile_solar.nc")

ONWIND_PROFILE_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/EGS_Test/renewable_profiles/profile_onwind.nc")

OFFWIND_AC_PROFILE_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/EGS_Test/renewable_profiles/profile_offwind-ac.nc")

OFFWIND_DC_PROFILE_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/EGS_Test/renewable_profiles/profile_offwind-dc.nc")

HYDRO_PROFILE_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/EGS_Test/renewable_profiles/profile_hydro.nc")

EGS_POTENTIAL_PATH = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/EGS_Test/egs_potentials_10_2050.csv")

# ------------------------------------------------------------
# Optional: Ausgabeordner für spätere Grafiken und Tabellen
# ------------------------------------------------------------

OUTPUT_DIR = Path("./outputs_scenario_comparison")

## 3. Hilfsfunktionen zum Laden von Dateien

Die folgenden Funktionen prüfen zuerst, ob eine Datei existiert. Dadurch bricht das Notebook nicht sofort ab, wenn optionale Dateien noch nicht eingetragen sind.

In [3]:
def path_exists(path):
    """Returns True if path is a valid existing file or directory."""
    if path is None:
        return False
    path = Path(path)
    return str(path).strip() != "" and path.exists()


def require_file(path, label):
    """Raises a clear error if a required file does not exist."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{label} wurde nicht gefunden:{path}"
            "Bitte den Pfad im zentralen Pfadblock prüfen."
        )
    return path


def load_yaml_optional(path):
    """Loads a YAML file if it exists. Otherwise returns None."""
    if not path_exists(path):
        return None

    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_geodata_optional(path):
    """Loads a GeoDataFrame if the file exists. Otherwise returns None."""
    if not path_exists(path):
        return None

    return gpd.read_file(path)


def load_xarray_optional(path):
    """Loads a NetCDF dataset if the file exists. Otherwise returns None."""
    if not path_exists(path):
        return None

    return xr.open_dataset(path)

## 4. Netzwerke laden

Nach diesem Block stehen beide Netzwerke als `n_a` und `n_b` zur Verfügung.

Zusätzlich wird ein Dictionary `networks` erzeugt. Das ist praktisch für spätere Schleifen über beide Szenarien.

In [4]:
# Pflichtdateien prüfen
NETWORK_PATH_SCENARIO_A = require_file(NETWORK_PATH_SCENARIO_A, "Netzwerk Szenario A")
NETWORK_PATH_SCENARIO_B = require_file(NETWORK_PATH_SCENARIO_B, "Netzwerk Szenario B")

# Netzwerke laden
n_a = pypsa.Network(NETWORK_PATH_SCENARIO_A)
n_b = pypsa.Network(NETWORK_PATH_SCENARIO_B)

# Praktische Container für spätere Vergleiche
networks = {
    SCENARIO_A_NAME: n_a,
    SCENARIO_B_NAME: n_b,
}

network_paths = {
    SCENARIO_A_NAME: NETWORK_PATH_SCENARIO_A,
    SCENARIO_B_NAME: NETWORK_PATH_SCENARIO_B,
}

print("Netzwerke erfolgreich geladen:")
for scenario_name, network_path in network_paths.items():
    print(f"- {scenario_name}: {network_path}")

Netzwerke erfolgreich geladen:
- EGS: /mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/EGS_Test/postnetworks/elec_s_10_ec_lcopt_Co2L0.0-Ep_144h_2050_0.0514_AG_0export_393import.nc
- without EGS: /mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/Without_EGS_Test/postnetworks/elec_s_10_ec_lcopt_Co2L0.0-Ep_144h_2050_0.0514_AG_0export_393import.nc


## 5. Optionale Zusatzdateien laden

Dieser Block lädt Configs, Geodaten und Ressourcenprofile, falls die Pfade existieren.

Wenn eine Datei nicht existiert, wird einfach `None` gespeichert. Dadurch kannst du später gezielt prüfen, ob die jeweilige Datei verfügbar ist.

In [5]:
# ------------------------------------------------------------
# Configs
# ------------------------------------------------------------

cfg_a = load_yaml_optional(CONFIG_PATH_SCENARIO_A)
cfg_b = load_yaml_optional(CONFIG_PATH_SCENARIO_B)

configs = {
    SCENARIO_A_NAME: cfg_a,
    SCENARIO_B_NAME: cfg_b,
}

# ------------------------------------------------------------
# Geodaten
# ------------------------------------------------------------

country_shapes = load_geodata_optional(COUNTRY_SHAPES_PATH)
gadm_shapes = load_geodata_optional(GADM_SHAPES_PATH)
gadm_shapes_10 = load_geodata_optional(GADM_SHAPES_PATH_10)
regions_onshore = load_geodata_optional(REGIONS_ONSHORE_PATH)
regions_offshore = load_geodata_optional(REGIONS_OFFSHORE_PATH)

geo_data = {
    "country_shapes": country_shapes,
    "gadm_shapes": gadm_shapes,
    "gadm_shapes_10": gadm_shapes_10,
    "regions_onshore": regions_onshore,
    "regions_offshore": regions_offshore,
}

# ------------------------------------------------------------
# Ressourcenprofile
# ------------------------------------------------------------

solar_profile = load_xarray_optional(SOLAR_PROFILE_PATH)
onwind_profile = load_xarray_optional(ONWIND_PROFILE_PATH)
offwind_ac_profile = load_xarray_optional(OFFWIND_AC_PROFILE_PATH)
offwind_dc_profile = load_xarray_optional(OFFWIND_DC_PROFILE_PATH)
hydro_profile = load_xarray_optional(HYDRO_PROFILE_PATH)

profiles = {
    "solar": solar_profile,
    "onwind": onwind_profile,
    "offwind_ac": offwind_ac_profile,
    "offwind_dc": offwind_dc_profile,
    "hydro": hydro_profile,
}

# ------------------------------------------------------------
# Ausgabeordner anlegen
# ------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Optionale Dateien geladen, soweit vorhanden.")
print(f"Ausgabeordner: {OUTPUT_DIR.resolve()}")

Optionale Dateien geladen, soweit vorhanden.
Ausgabeordner: /mnt/c/Users/nikla/Documents/Git/EGS-Analyse-Notebooks/Notebooks/outputs_scenario_comparison


## 6. Plausibilitätsübersicht

Dieser Block zeigt eine kurze Übersicht beider Netzwerke. So kannst du direkt prüfen, ob die richtigen Dateien geladen wurden.

In [6]:
def network_overview(network, scenario_name):
    """Creates a compact overview table for a PyPSA network."""
    objective = getattr(network, "objective", np.nan)

    return {
        "Szenario": scenario_name,
        "Snapshots": len(network.snapshots),
        "Start": network.snapshots.min() if len(network.snapshots) > 0 else None,
        "Ende": network.snapshots.max() if len(network.snapshots) > 0 else None,
        "Buses": len(network.buses),
        "Generators": len(network.generators),
        "Links": len(network.links),
        "Stores": len(network.stores),
        "Storage Units": len(network.storage_units),
        "Loads": len(network.loads),
        "Lines": len(network.lines),
        "Objective": objective,
    }


overview = pd.DataFrame(
    [
        network_overview(n, scenario_name)
        for scenario_name, n in networks.items()
    ]
)

display(overview)

,Szenario,Snapshots,Start,Ende,Buses,Generators,Links,Stores,Storage Units,Loads,Lines,Objective
0,EGS,61,2013-01-01,2013-12-27,1182,1864,3199,754,91,1421,88,1.009667e+11
1,without EGS,61,2013-01-01,2013-12-27,1134,1815,3058,754,44,1421,88,1.054113e+11


## 7. Verfügbare Carrier prüfen

Dieser Block zeigt, welche Carrier in beiden Netzwerken vorkommen. Das ist hilfreich, bevor später technologiespezifische Auswertungen erstellt werden.

In [7]:
def collect_carriers(network):
    """Collects carriers from the most relevant PyPSA component tables."""
    carrier_parts = []

    for component_name in ["buses", "generators", "links", "stores", "storage_units", "loads"]:
        table = getattr(network, component_name, None)

        if table is None or table.empty or "carrier" not in table.columns:
            continue

        carriers = (
            table["carrier"]
            .dropna()
            .astype(str)
            .replace("", np.nan)
            .dropna()
            .unique()
            .tolist()
        )

        carrier_parts.extend(carriers)

    return sorted(set(carrier_parts))


carrier_overview = pd.DataFrame({
    scenario_name: pd.Series(collect_carriers(network))
    for scenario_name, network in networks.items()
})

display(carrier_overview)

,EGS,without EGS
0,AC,AC
1,Alkaline electrolyzer large,Alkaline electrolyzer large
2,B2B,B2B
3,BEV charger,BEV charger
4,CCGT,CCGT
...,...,...
132,urban central solid biomass CHP,urban central water tanks charger
133,urban central solid biomass CHP CC,urban central water tanks discharger
134,urban central water tanks,NaN
135,urban central water tanks charger,NaN


In [8]:
# ============================================================
# EGS-Gringarten-Daten für Japan nach GADM-Regionen aggregieren
#
# Input:
# /mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/data/egs_data/egs_input_dataset.csv
#
# Output:
# /mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/data/egs_data/egs_gringarten_japan_gadm1.csv
#
# Betrachtet wird nur:
# - Pout_Gringarten_MW
# - LCOE_Gringarten_Eurct_per_kWh
#
# Wichtig:
# Die LCOE-Spalte wird als EUR/kWh interpretiert, nicht als Cent/kWh.
# ============================================================

import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
from pathlib import Path


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

EGS_INPUT_DATASET_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/data/egs_data/egs_input_dataset.csv"
)

OUTPUT_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/data/egs_data/egs_gringarten_japan_gadm1.csv"
)

# Falls GADM_SHAPES_PATH in deinem Notebook schon definiert ist, wird dieser verwendet.
# Sonst wird dieser Standardpfad genutzt.
GADM_PATH = Path(
    GADM_SHAPES_PATH
    if "GADM_SHAPES_PATH" in globals()
    else "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/data/gadm/gadm41_JPN/gadm41_JPN.gpkg"
)

gadm_level = 1

country_filter = "JPN"

pout_col = "Pout_Gringarten_MW"
lcoe_col_original = "LCOE_Gringarten_Eurct_per_kWh"

# Neuer klarer Name, weil die Werte EUR/kWh sind, nicht Cent/kWh.
lcoe_col = "LCOE_Gringarten_EUR_per_kWh"


# ------------------------------------------------------------
# Helper: GADM-Layer robust laden
# ------------------------------------------------------------

def find_gadm_layer(gpkg_path, gadm_level):
    """
    Findet automatisch den passenden GADM-Layer in einem GeoPackage.
    Funktioniert für Layernamen wie:
    - ADM_ADM_1
    - gadm41_JPN — ADM_ADM_1
    """
    gpkg_path = Path(gpkg_path)

    target = f"ADM_ADM_{gadm_level}"
    layers = fiona.listlayers(str(gpkg_path))

    matches = [
        layer for layer in layers
        if layer.endswith(target) or target in layer
    ]

    if not matches:
        raise ValueError(
            f"Kein Layer für {target} in {gpkg_path} gefunden. "
            f"Vorhandene Layer: {layers}"
        )

    return matches[0]


def read_gadm_layer(gpkg_path, gadm_level):
    """
    Liest einen bestimmten GADM-Layer aus einem GeoPackage.
    """
    gpkg_path = Path(gpkg_path)

    if gpkg_path.suffix.lower() == ".gpkg":
        layer = find_gadm_layer(gpkg_path, gadm_level)
        gdf = gpd.read_file(gpkg_path, layer=layer)
    else:
        gdf = gpd.read_file(gpkg_path)

    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    return gdf.to_crs("EPSG:4326")


def normalize_region_id(region_id):
    """
    Harmonisiert GADM-IDs mit PyPSA-Earth-Regionen.

    Beispiel:
    JPN.13_1 -> JP.13_1
    """
    s = str(region_id).strip()

    if s.startswith("JPN."):
        s = "JP." + s[len("JPN."):]

    return s


def infer_region_id_col(gdf, gadm_level=1):
    candidates = [
        f"GID_{gadm_level}",
        "GADM_ID",
        "region",
        "GID_2",
        "GID_1",
        "NAME_2",
        "NAME_1",
        "name",
        "Name",
    ]

    for col in candidates:
        if col in gdf.columns:
            return col

    raise ValueError(
        "Keine passende Regions-ID-Spalte gefunden. "
        f"Vorhandene Spalten: {list(gdf.columns)}"
    )


def sjoin_compat(left, right, how="left", predicate="within"):
    """
    Kompatibilität für verschiedene GeoPandas-Versionen.
    """
    try:
        return gpd.sjoin(
            left,
            right,
            how=how,
            predicate=predicate
        )
    except TypeError:
        return gpd.sjoin(
            left,
            right,
            how=how,
            op=predicate
        )


# ------------------------------------------------------------
# 1) EGS-Datensatz einlesen
# ------------------------------------------------------------

expected_columns = [
    "lon",
    "lat",
    "gid1",
    "Pout_Gringarten_MW",
    "LCOE_Gringarten_Eurct_per_kWh",
    "Pout_VolumeMethod_MW",
    "LCOE_VolumeMethod_Eurct_per_kWh",
    "Pout_Sustainable_MW",
    "LCOE_Sustainable_Eurct_per_kWh",
]

df = pd.read_csv(EGS_INPUT_DATASET_PATH)

# Falls die Datei keinen Header hat, nochmal mit festen Spaltennamen laden
if not set(["lon", "lat", "gid1"]).issubset(df.columns):
    df = pd.read_csv(
        EGS_INPUT_DATASET_PATH,
        header=None,
        names=expected_columns
    )

required_cols = ["lon", "lat", "gid1", pout_col, lcoe_col_original]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Diese benötigten Spalten fehlen im EGS-Datensatz: {missing_cols}\n"
        f"Vorhandene Spalten: {list(df.columns)}"
    )

# Nur relevante Gringarten-Spalten behalten
df = df[
    [
        "lon",
        "lat",
        "gid1",
        pout_col,
        lcoe_col_original,
    ]
].copy()

df = df.rename(columns={
    lcoe_col_original: lcoe_col
})

# Datentypen bereinigen
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df[pout_col] = pd.to_numeric(df[pout_col], errors="coerce")
df[lcoe_col] = pd.to_numeric(df[lcoe_col], errors="coerce")

df = df.dropna(subset=["lon", "lat", "gid1", pout_col, lcoe_col])


# ------------------------------------------------------------
# 2) Japan herausfiltern
# ------------------------------------------------------------

gid_clean = (
    df["gid1"]
    .astype(str)
    .str.strip()
    .str.upper()
)

japan_df = df[
    gid_clean.eq(country_filter)
    | gid_clean.str.startswith(f"{country_filter}.")
].copy()

if japan_df.empty:
    raise ValueError(
        f"Keine Zeilen für Japan gefunden. Erwarteter gid1-Wert: {country_filter}"
    )

print("------------------------------------------------------------")
print("EGS-Gringarten-Daten Japan")
print("------------------------------------------------------------")
print(f"Anzahl Punkte Japan: {len(japan_df):,}")
print(f"Gesamte Gringarten-Leistung vor GADM-Verschnitt: {japan_df[pout_col].sum():,.2f} MW")
print(f"Einfacher mittlerer LCOE: {japan_df[lcoe_col].mean():.4f} EUR/kWh")


# ------------------------------------------------------------
# 3) Punkte als GeoDataFrame erzeugen
# ------------------------------------------------------------

points = gpd.GeoDataFrame(
    japan_df,
    geometry=gpd.points_from_xy(
        japan_df["lon"],
        japan_df["lat"]
    ),
    crs="EPSG:4326"
)


# ------------------------------------------------------------
# 4) GADM-Regionen laden
# ------------------------------------------------------------

gadm = read_gadm_layer(
    GADM_PATH,
    gadm_level
)

region_id_col = infer_region_id_col(
    gadm,
    gadm_level=gadm_level
)

gadm["region_original"] = gadm[region_id_col].astype(str).str.strip()
gadm["region"] = gadm["region_original"].map(normalize_region_id)

# Optionale Namensspalte
name_col = None
for candidate in [f"NAME_{gadm_level}", "NAME_1", "name", "Name"]:
    if candidate in gadm.columns:
        name_col = candidate
        break

keep_cols = ["region", "region_original", "geometry"]

if name_col is not None:
    gadm["region_name"] = gadm[name_col].astype(str)
    keep_cols.insert(2, "region_name")
else:
    gadm["region_name"] = gadm["region"]
    keep_cols.insert(2, "region_name")

gadm = gadm[keep_cols].copy()

print("")
print("------------------------------------------------------------")
print("GADM-Regionen")
print("------------------------------------------------------------")
print(f"GADM-Level: {gadm_level}")
print(f"Verwendete ID-Spalte: {region_id_col}")
print(f"Anzahl Regionen: {len(gadm)}")
print(gadm[["region", "region_name"]].head(10).to_string(index=False))


# ------------------------------------------------------------
# 5) Punkte mit GADM-Regionen verschneiden
# ------------------------------------------------------------

joined = sjoin_compat(
    points,
    gadm[["region", "region_original", "region_name", "geometry"]],
    how="left",
    predicate="within"
)

# Nicht zugeordnete Punkte anzeigen
unmatched = joined[joined["region"].isna()].copy()

if not unmatched.empty:
    print("")
    print("Warnung: Einige Punkte konnten keiner GADM-Region zugeordnet werden.")
    print(f"Nicht zugeordnete Punkte: {len(unmatched)}")
    print(unmatched[["lon", "lat", "gid1", pout_col, lcoe_col]].head(20).to_string(index=False))

# Nur zugeordnete Punkte weiterverwenden
joined = joined.dropna(subset=["region"]).copy()


# ------------------------------------------------------------
# 6) Regionale Aggregation
# ------------------------------------------------------------
# Durchschnittlicher LCOE:
# - lcoe_mean_EUR_per_kWh: einfacher Mittelwert über Punkte
# - lcoe_weighted_by_power_EUR_per_kWh: leistungsgewichteter Mittelwert
#
# Für die spätere Modellierung ist der leistungsgewichtete Wert meist sinnvoller,
# weil Punkte mit höherer Leistung stärker berücksichtigt werden.
# ------------------------------------------------------------

def weighted_average_lcoe(group):
    weights = group[pout_col].fillna(0)

    if weights.sum() <= 0:
        return group[lcoe_col].mean()

    return np.average(
        group[lcoe_col],
        weights=weights
    )


regional = (
    joined
    .groupby(["region", "region_original", "region_name"], as_index=False)
    .agg(
        n_points=("geometry", "count"),
        Pout_Gringarten_MW=(pout_col, "sum"),
        LCOE_Gringarten_mean_EUR_per_kWh=(lcoe_col, "mean"),
        LCOE_Gringarten_min_EUR_per_kWh=(lcoe_col, "min"),
        LCOE_Gringarten_max_EUR_per_kWh=(lcoe_col, "max"),
    )
)

weighted_lcoe = (
    joined
    .groupby(["region", "region_original", "region_name"])
    .apply(weighted_average_lcoe)
    .rename("LCOE_Gringarten_weighted_EUR_per_kWh")
    .reset_index()
)

regional = regional.merge(
    weighted_lcoe,
    on=["region", "region_original", "region_name"],
    how="left"
)

# Zusätzliche Einheiten für bessere Vergleichbarkeit
regional["Pout_Gringarten_GW"] = (
    regional["Pout_Gringarten_MW"] / 1e3
)

regional["LCOE_Gringarten_mean_EUR_per_MWh"] = (
    regional["LCOE_Gringarten_mean_EUR_per_kWh"] * 1e3
)

regional["LCOE_Gringarten_weighted_EUR_per_MWh"] = (
    regional["LCOE_Gringarten_weighted_EUR_per_kWh"] * 1e3
)

regional = regional[
    [
        "region",
        "region_original",
        "region_name",
        "n_points",
        "Pout_Gringarten_MW",
        "Pout_Gringarten_GW",
        "LCOE_Gringarten_mean_EUR_per_kWh",
        "LCOE_Gringarten_weighted_EUR_per_kWh",
        "LCOE_Gringarten_mean_EUR_per_MWh",
        "LCOE_Gringarten_weighted_EUR_per_MWh",
        "LCOE_Gringarten_min_EUR_per_kWh",
        "LCOE_Gringarten_max_EUR_per_kWh",
    ]
].copy()

regional = regional.sort_values(
    "Pout_Gringarten_MW",
    ascending=False
)


# ------------------------------------------------------------
# 7) CSV speichern
# ------------------------------------------------------------

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

regional.to_csv(
    OUTPUT_PATH,
    index=False
)

print("")
print("------------------------------------------------------------")
print("Output gespeichert")
print("------------------------------------------------------------")
print(OUTPUT_PATH)

print("")
print("Zusammenfassung:")
print(f"Regionen mit EGS-Punkten: {len(regional)}")
print(f"Gesamtleistung regional aggregiert: {regional['Pout_Gringarten_MW'].sum():,.2f} MW")
print(f"Gesamtleistung regional aggregiert: {regional['Pout_Gringarten_GW'].sum():,.2f} GW")

display(regional.head(20))

------------------------------------------------------------
EGS-Gringarten-Daten Japan
------------------------------------------------------------
Anzahl Punkte Japan: 20,665
Gesamte Gringarten-Leistung vor GADM-Verschnitt: 137,994.14 MW
Einfacher mittlerer LCOE: 0.1062 EUR/kWh

------------------------------------------------------------
GADM-Regionen
------------------------------------------------------------
GADM-Level: 1
Verwendete ID-Spalte: GID_1
Anzahl Regionen: 47
 region region_name
 JP.1_1       Aichi
 JP.2_1       Akita
 JP.3_1      Aomori
 JP.4_1       Chiba
 JP.5_1       Ehime
 JP.6_1       Fukui
 JP.7_1     Fukuoka
 JP.8_1   Fukushima
 JP.9_1        Gifu
JP.10_1       Gunma

------------------------------------------------------------
Output gespeichert
------------------------------------------------------------
/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/data/egs_data/egs_gringarten_japan_gadm1.csv

Zusammenfassung:
Regionen mit EGS-Punkten: 47
Gesamtleis

,region,region_original,region_name,n_points,Pout_Gringarten_MW,Pout_Gringarten_GW,LCOE_Gringarten_mean_EUR_per_kWh,LCOE_Gringarten_weighted_EUR_per_kWh,LCOE_Gringarten_mean_EUR_per_MWh,LCOE_Gringarten_weighted_EUR_per_MWh,LCOE_Gringarten_min_EUR_per_kWh,LCOE_Gringarten_max_EUR_per_kWh
2,JP.12_1,JPN.12_1,Hokkaido,7375,37895.447033,37.895447,0.124775,0.102752,124.775375,102.751748,0.065293,0.260664
8,JP.18_1,JPN.18_1,Kagoshima,510,15422.474041,15.422474,0.031443,0.030903,31.442502,30.902900,0.030014,0.040752
21,JP.2_1,JPN.2_1,Akita,873,8999.323961,8.999324,0.057016,0.055853,57.016315,55.852526,0.049679,0.088893
32,JP.3_1,JPN.3_1,Aomori,869,7288.344181,7.288344,0.073768,0.065786,73.767857,65.786416,0.049679,0.099368
12,JP.21_1,JPN.21_1,Kumamoto,345,6098.795636,6.098796,0.042012,0.039840,42.011650,39.840310,0.036345,0.058161
6,JP.16_1,JPN.16_1,Iwate,1155,5442.943626,5.442944,0.131573,0.111393,131.573238,111.393330,0.049679,0.170877
45,JP.8_1,JPN.8_1,Fukushima,921,4657.631600,4.657632,0.109702,0.101890,109.702056,101.889790,0.088353,0.190411
16,JP.25_1,JPN.25_1,Miyazaki,271,4569.771875,4.569772,0.042955,0.040891,42.955253,40.891500,0.030014,0.053679
38,JP.45_1,JPN.45_1,Yamagata,620,3934.675751,3.934676,0.084819,0.082138,84.819110,82.137896,0.058945,0.155277
20,JP.29_1,JPN.29_1,Niigata,667,3175.977167,3.175977,0.112627,0.107493,112.626784,107.493045,0.078572,0.155277
